In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score, accuracy_score
import joblib
import warnings
warnings.filterwarnings('ignore')

url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0})

if 'customerID' in df.columns:
    df.drop('customerID', axis=1, inplace=True)

X = df.drop('Churn', axis=1)
y = df['Churn']

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(include=[np.number]).columns.tolist()

print("Категориальные признаки:", cat_cols)
print("Числовые признаки:", num_cols)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
                                                    random_state=42, stratify=y)

# для числовых: заполнение медианой (на всякий случай) + масштабирование
# для категориальных: заполнение самым частым(мода) + OneHotEncoder
num_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_pipeline, num_cols),
        ('cat', cat_pipeline, cat_cols)
    ])

Категориальные признаки: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
Числовые признаки: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']


In [2]:
# модели для эксперимента 
models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest': RandomForestClassifier(n_estimators=100, random_state=42),
    'MLPClassifier': MLPClassifier(hidden_layer_sizes=(50, 30), max_iter=500, random_state=42)
}

# кросс-валидация для сравнения
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}
for name, model in models.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    # оцениваем по ROC-AUC, F1, Accuracy
    auc_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')
    f1_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='f1')
    acc_scores = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='accuracy')
    
    results[name] = {
        'ROC-AUC': auc_scores.mean(),
        'F1': f1_scores.mean(),
        'Accuracy': acc_scores.mean()
    }
    print(f"{name}: ROC-AUC={auc_scores.mean():.4f} (+/- {auc_scores.std():.4f}), "
          f"F1={f1_scores.mean():.4f}, Accuracy={acc_scores.mean():.4f}")

# вывод результатов сравнения
results_df = pd.DataFrame(results).T
results_df

LogisticRegression: ROC-AUC=0.8461 (+/- 0.0052), F1=0.5953, Accuracy=0.8025
RandomForest: ROC-AUC=0.8239 (+/- 0.0088), F1=0.5569, Accuracy=0.7916
MLPClassifier: ROC-AUC=0.7618 (+/- 0.0120), F1=0.4910, Accuracy=0.7378


,ROC-AUC,F1,Accuracy
LogisticRegression,0.846059,0.595262,0.802489
RandomForest,0.823946,0.556935,0.791644
MLPClassifier,0.761772,0.490967,0.737778


In [5]:
best_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42))
])
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)
y_proba = best_model.predict_proba(X_test)[:, 1]
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))
print("Test F1:", f1_score(y_test, y_pred))
print("Test Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

import joblib
joblib.dump(best_model, 'artifacts/model_pipeline.pkl')

Test ROC-AUC: 0.8359290473207676
Test F1: 0.6079545454545454
Test Accuracy: 0.8038379530916845
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1033
           1       0.65      0.57      0.61       374

    accuracy                           0.80      1407
   macro avg       0.75      0.73      0.74      1407
weighted avg       0.80      0.80      0.80      1407



['artifacts/model_pipeline.pkl']